In [23]:
import pandas as pd
import pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from geoai.utils_ds.preprocessing_ops import PreProcessingOperations
from geoai.utils_ml.model_ops import ModelOperations
from geoai.utils_geo.raster_ops import RasterOperations

preprocess_ops = PreProcessingOperations()
model_ops = ModelOperations()
raster_ops = RasterOperations()


# We will create a pipeline using the following steps:

1. **Load the data containing only the bands.**

1. **Compute indices**

1. **Bin and Categorize NDVI**

1. **Pipeline 1**

- `One hot encode NDVI_binary`

- `Ordinal encode NDVI_category`

- `Apply Log transformation to numerical features`

- `Do a Polynomial transformation`

5. **Pipeline 2**

- `Scale to 0-1`

- `Apply LDA`

- `Train a logisitic regression`


#### Step 1

In [24]:

X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR
0,350.0,542.0000,323.0,3277.0000,1975.3334
1,390.0,555.0000,380.0,3016.6667,1991.0000
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000
3,363.2,546.5000,395.0,3244.5000,2052.0000
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000


#### Step 2

In [25]:
X_train = raster_ops.indices_binary_category(X_train) 
X_test = raster_ops.indices_binary_category(X_test)
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545,high_veg,veg
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227,high_veg,veg
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127,low_veg,non_veg
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438,high_veg,veg
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098,low_veg,non_veg


#### Step 4: Pipeline 1

In [26]:
# for categorical features
one_hot_encoder_columns = ["NDVI_binary"]
ordinal_encoder_columns = ["NDVI_categorized"]
categories = [["low_veg", "medium_veg", "high_veg"]]
one_hot_transformer = OneHotEncoder(dtype=int, sparse_output=False) # Instantiate the one hot transformer
ordinal_transformer = OrdinalEncoder(categories=categories, dtype=int) # Instantiate the ordinal transformer

# for numerical features
numerical_columns = X_train.select_dtypes(include=["float64"]).columns.tolist()
log_trasformer = FunctionTransformer(func=np.log1p)  # Instantiate the log transformer
poly_transformer = PolynomialFeatures(
    degree=2
)  # Instantiate the polynomial transformer


# make a pipeline for each type of transformer
categorical_transformer_1 = Pipeline(steps=[
    ('one_hot_transformer', one_hot_transformer)
])

categorical_transformer_2 = Pipeline(steps=[
    ('ordinal_transformer', ordinal_transformer)
])

numerical_transformer = Pipeline(steps=[
    ('log', log_trasformer),
    ('poly', poly_transformer)
])

# Create a preprocessor that includes the numerical, one hot, and ordinal transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical_transformer_1', categorical_transformer_1, one_hot_encoder_columns),
        ('categorical_transformer_2', categorical_transformer_2, ordinal_encoder_columns),
        ('numerical_transformer', numerical_transformer, numerical_columns)
    ])

#### Step 5: Pipeline 2

In [27]:
# Create the final pipeline
min_max_scaler = MinMaxScaler()
lda = LinearDiscriminantAnalysis(n_components=3)
classifier = LogisticRegression(solver='lbfgs', max_iter=10000, random_state=1) 

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('min_max_scaler', min_max_scaler),
    ('lda', lda),  
    ('classifier', classifier)
])
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                  Pipeline(steps=[('one_hot_transformer',
                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                 sparse_output=False))]),
                                                  ['NDVI_binary']),
                                                 ('categorical_transformer_2',
                                                  Pipeline(steps=[('ordinal_transformer',
                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                               'medium_veg',
                                                                                               'high_veg']],
                                                                                  dtype=<class...
                                                  ['NDVI_categorized']),
                                                 ('numerical_transformer',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('poly',
                                                                   PolynomialFeatures())]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI'])])),
                ('min_max_scaler', MinMaxScaler()),
                ('lda', LinearDiscriminantAnalysis(n_components=3)),
                ('classifier',
                 LogisticRegression(max_iter=10000, random_state=1))])

In [28]:
# Fit the model. Fitting means training the model
# The fit method taked the training X and y as input 
pipeline.fit(X_train, y_train.values.ravel())

# Predict the training and test data to get the predicted y values for the
# training and test data
y_train_pred = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

# Calculate the accuracy for both the training and test data
# We want to have a high accuracy (near 1) for both the training and test data
# and the difference between the two accuracies should be small 
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)[3]}")
print(f"Test Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_pred_test)[3]}")

Train Accuracy: 0.9367562648316211
Test Accuracy: 0.9290355525568245


In [29]:
# save the model using pickle
# merge the train and test datasets
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the whole dataset
pipeline.fit(X_all, y_all)
with open('trained_models/lda_pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


END